# Experimento multimodal: fusión por tokens con attention pooling

Clasificación binaria de morfología MaNGA (`temprana=0`, `tardia=1`) con imágenes RGB Legacy Survey y espectros `FLUX_FoV`.

Variantes comparables:

- **M0** `independent_mean`: mean pooling independiente y concatenación.
- **M1** `independent_attention`: attention pooling enmascarado independiente y concatenación.
- **M2** `joint_attention`: attention pooling conjunto sobre todos los tokens.

El notebook no ejecuta entrenamiento automáticamente. Para entrenar, revisar la configuración y cambiar `RUN_TRAINING` a `True`.

In [1]:
from __future__ import annotations

import copy
import json
import random
import time
from pathlib import Path
from typing import Any, Mapping

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent.parent.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dataloader.multimodal import build_dataloaders
from src.evaluate import evaluate_binary_classifier
from src.models import (
    build_multimodal_token_fusion,
    load_spectral_encoder_checkpoint,
    load_visual_encoder_checkpoint,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Proyecto: {ROOT}")
print(f"Dispositivo: {DEVICE}")

/home/albertor/Documents/deep_learning/repos/clasificacion-galaxias/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Proyecto: /home/albertor/Documents/deep_learning/repos/clasificacion-galaxias
Dispositivo: cuda


## Configuración

Los paths y los hiperparámetros replican las convenciones de los experimentos unimodales existentes. `fusion_variant` selecciona M0, M1 o M2.

In [13]:
CONFIG: dict[str, Any] = {
    "experiment_name": "multimodal_token_fusion_attention_m2",
    "seed": 42,
    "data": {
        "project_root": str(ROOT),
        "manifest": "data/galaxy_manifest.csv",
        "splits": "data/processed/splits_seed42_obj-label_FoV.csv",
        "modalities": ["image", "spectrum"],
        "image": {"fov_column": "fov_0", "size": 256, "normalization": "none"},
        "spectrum": {"cache_dir": "data/processed/spectra/spectra_cache_FoV"},
    },
    "model": {
        "name": "multimodal_token_fusion",
        "fusion_variant": "joint_attention",  # M0, M1 o M2
        "token_dim": 128,
        "classifier_hidden_dim": 128,
        "dropout": 0.3,
        "patch_size": 32,
        "gru_hidden": 64,
        "gru_layers": 2,
        "layer_norm": True,
        "visual_positional_encoding": False,
        "freeze_visual_encoder": True,
        "freeze_spectral_encoder": True,
    },
    "checkpoints": {
        "visual": "models/experiments/imagenes_resnet18_fov/best_model.pt",
        "spectral": "models/experiments/espectros_fov_patch_bigru/best_model.pt",
    },
    "training": {
        "batch_size": 32,
        "num_workers": 0,
        "pin_memory": DEVICE.type == "cuda",
        "epochs": 50,
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
        "early_stopping_patience": 8,
        "max_grad_norm": 1.0,
    },
    "selection_metric": "roc_auc",
}

RUN_SMOKE_TEST = False
RUN_TRAINING = True
RUN_EVALUATION = True

RUN_DIR = ROOT / "models" / "experiments" / CONFIG["experiment_name"]
CHECKPOINT_PATH = RUN_DIR / "best_model.pt"

In [14]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def move_batch_to_device(batch: Mapping[str, Any], device: torch.device) -> dict[str, Any]:
    return {
        key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value
        for key, value in batch.items()
    }


def count_parameters(model: nn.Module) -> tuple[int, int, int]:
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable, total - trainable


def to_jsonable(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Mapping):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    return value


set_seed(CONFIG["seed"])

## DataLoaders y modelo

El DataLoader multimodal existente entrega un diccionario con `image`, `spectrum`, `spectrum_mask`, `label` e `id`. Los checkpoints se cargan de forma estricta: cualquier clave faltante o incompatible produce un error.

In [15]:
train_loader, validation_loader, test_loader = build_dataloaders(CONFIG)
sample_batch = next(iter(train_loader))

model = build_multimodal_token_fusion(CONFIG, sample_batch)
visual_checkpoint = ROOT / CONFIG["checkpoints"]["visual"]
spectral_checkpoint = ROOT / CONFIG["checkpoints"]["spectral"]
if not visual_checkpoint.is_file():
    raise FileNotFoundError(f"No existe el checkpoint visual: {visual_checkpoint}")
if not spectral_checkpoint.is_file():
    raise FileNotFoundError(f"No existe el checkpoint espectral: {spectral_checkpoint}")

visual_report = load_visual_encoder_checkpoint(model.visual_encoder, visual_checkpoint)
spectral_report = load_spectral_encoder_checkpoint(model.spectral_encoder, spectral_checkpoint)
model = model.to(DEVICE)

total, trainable, frozen = count_parameters(model)
print(f"Modelo: {CONFIG['model']['fusion_variant']}")
print(f"Parámetros total={total:,}; entrenables={trainable:,}; congelados={frozen:,}")
print(f"Checkpoint visual: {len(visual_report.loaded_keys)} claves cargadas")
print(f"Checkpoint espectral: {len(spectral_report.loaded_keys)} claves cargadas")

Modelo: joint_attention
Parámetros total=11,462,852; entrenables=99,972; congelados=11,362,880
Checkpoint visual: 120 claves cargadas
Checkpoint espectral: 22 claves cargadas


In [16]:
if RUN_SMOKE_TEST:
    model.eval()
    batch_device = move_batch_to_device(sample_batch, DEVICE)
    with torch.no_grad():
        visual_tokens, visual_mask, spectral_tokens, spectral_mask = model._project_tokens(batch_device)
        logits = model(batch_device)

    assert logits.shape == batch_device["label"].shape
    assert torch.isfinite(logits).all()
    assert visual_mask.all()
    assert spectral_mask.any(dim=1).all()
    print(f"Tokens visuales: {tuple(visual_tokens.shape)}")
    print(f"Tokens espectrales: {tuple(spectral_tokens.shape)}")
    print(f"Logits: {tuple(logits.shape)}")

    if CONFIG["model"]["fusion_variant"] != "independent_mean":
        weights = model.get_attention_weights(batch_device)
        for name, value in weights.items():
            if name != "visual_token_count":
                assert torch.isfinite(value).all()
                print(f"Pesos {name}: {tuple(value.shape)}")

## Entrenamiento

Esta celda define el loop, pero no inicia un entrenamiento mientras `RUN_TRAINING` siga en `False`. El mejor estado se selecciona por ROC-AUC de validación.

In [17]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
) -> tuple[float, np.ndarray, np.ndarray]:
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    labels, probabilities = [], []

    for batch in tqdm(loader, leave=False, desc="train" if is_training else "validation"):
        batch = move_batch_to_device(batch, DEVICE)
        with torch.set_grad_enabled(is_training):
            logits = model(batch)
            loss = criterion(logits, batch["label"])
            if is_training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["training"]["max_grad_norm"])
                optimizer.step()
        total_loss += loss.item() * batch["label"].numel()
        labels.append(batch["label"].detach().cpu().numpy())
        probabilities.append(torch.sigmoid(logits).detach().cpu().numpy())

    return total_loss / len(loader.dataset), np.concatenate(labels), np.concatenate(probabilities)


def fit(model: nn.Module) -> pd.DataFrame:
    training = CONFIG["training"]
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=training["learning_rate"],
        weight_decay=training["weight_decay"],
    )
    best_score = float("-inf")
    best_state: dict[str, torch.Tensor] | None = None
    patience = 0
    history: list[dict[str, float]] = []

    for epoch in range(1, training["epochs"] + 1):
        started_at = time.perf_counter()
        train_loss, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        validation_loss, y_true, y_proba = run_epoch(model, validation_loader, criterion)
        y_pred = (y_proba >= 0.5).astype(np.int64)
        metrics = evaluate_binary_classifier(y_true, y_pred, y_proba)
        score = float(metrics[CONFIG["selection_metric"]])
        if not np.isfinite(score):
            raise FloatingPointError("La métrica de selección no es finita.")

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_loss,
            "validation_accuracy": float(metrics["accuracy"]),
            "validation_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "validation_f1": f1_score(y_true, y_pred, zero_division=0),
            "validation_roc_auc": float(metrics["roc_auc"]),
            "seconds": time.perf_counter() - started_at,
        })
        print(f"[{epoch:02d}/{training['epochs']}] train_loss={train_loss:.4f} val_loss={validation_loss:.4f} val_auc={score:.4f}")

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= training["early_stopping_patience"]:
                print("Early stopping por validación.")
                break

    if best_state is None:
        raise RuntimeError("No se obtuvo un checkpoint de validación.")
    model.load_state_dict(best_state)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    torch.save({"model_state_dict": best_state, "config": to_jsonable(CONFIG)}, CHECKPOINT_PATH)
    return pd.DataFrame(history)

In [18]:
if RUN_TRAINING:
    history = fit(model)
    history.to_csv(RUN_DIR / "history.csv", index=False)
    (RUN_DIR / "config.json").write_text(json.dumps(to_jsonable(CONFIG), indent=2), encoding="utf-8")
    history.tail()

[01/50] train_loss=0.2544 val_loss=0.2658 val_auc=0.9630


[02/50] train_loss=0.1879 val_loss=0.2531 val_auc=0.9639


[03/50] train_loss=0.1806 val_loss=0.2572 val_auc=0.9648


[04/50] train_loss=0.1786 val_loss=0.2576 val_auc=0.9646


[05/50] train_loss=0.1714 val_loss=0.2632 val_auc=0.9652


[06/50] train_loss=0.1769 val_loss=0.2491 val_auc=0.9654


... (21 outputs left out)

[28/50] train_loss=0.1579 val_loss=0.2495 val_auc=0.9652


[29/50] train_loss=0.1577 val_loss=0.2578 val_auc=0.9647


[30/50] train_loss=0.1512 val_loss=0.2585 val_auc=0.9651


[31/50] train_loss=0.1530 val_loss=0.2650 val_auc=0.9651
Early stopping por validación.


## Evaluación de test y exportación de artefactos

Activa esta sección únicamente después de entrenar o de cargar un checkpoint compatible. Muestra ROC-AUC y el reporte de clasificación —incluida precisión por clase— sobre test. También guarda `results.json`, `test_metrics.json` y `predictions.csv`, igual que los experimentos espectrales de control. Los pesos de atención son diagnósticos; no constituyen una explicación causal de la predicción.

In [19]:
def predict_split(model: nn.Module, loader: DataLoader, split_name: str) -> pd.DataFrame:
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False, desc=f"predict {split_name}"):
            ids = list(batch["id"])
            batch = move_batch_to_device(batch, DEVICE)
            probabilities = torch.sigmoid(model(batch)).cpu().numpy()
            labels = batch["label"].cpu().numpy().astype(np.int64)
            rows.append(pd.DataFrame({
                "name": ids,
                "split": split_name,
                "y_true": labels,
                "y_proba": probabilities,
                "y_pred": (probabilities >= 0.5).astype(np.int64),
            }))
    return pd.concat(rows, ignore_index=True)


if RUN_EVALUATION:
    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"No existe el checkpoint del experimento: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    criterion = nn.BCEWithLogitsLoss()
    test_loss, y_test, y_test_proba = run_epoch(model, test_loader, criterion)
    y_test_pred = (y_test_proba >= 0.5).astype(np.int64)
    test_metrics = evaluate_binary_classifier(y_test, y_test_pred, y_test_proba)
    results = {"test_loss": test_loss, **test_metrics}
    print(
        f"test_loss={test_loss:.4f}  test_accuracy={test_metrics['accuracy']:.4f}  "
        f"test_auc={test_metrics['roc_auc']:.4f}"
    )
    display(pd.DataFrame(test_metrics["report"]).T)

    predictions = pd.concat([
        predict_split(model, train_loader, "train"),
        predict_split(model, validation_loader, "validation"),
        predict_split(model, test_loader, "test"),
    ], ignore_index=True)
    predictions["label"] = predictions["y_true"].map({0: "temprana", 1: "tardia"})
    predictions["pred_label"] = predictions["y_pred"].map({0: "temprana", 1: "tardia"})
    predictions["correct"] = predictions["y_true"].eq(predictions["y_pred"])
    predictions = predictions[["name", "split", "label", "y_true", "y_proba", "y_pred", "pred_label", "correct"]]

    RUN_DIR.mkdir(parents=True, exist_ok=True)
    predictions.to_csv(RUN_DIR / "predictions.csv", index=False)
    (RUN_DIR / "results.json").write_text(json.dumps(to_jsonable(results), indent=2), encoding="utf-8")
    (RUN_DIR / "test_metrics.json").write_text(json.dumps(to_jsonable(results), indent=2), encoding="utf-8")
    print(f"Resultados guardados en: {RUN_DIR}")
    predictions.head()


validation:   0%|          | 0/44 [00:00<?, ?it/s]

test_loss=0.2479  test_accuracy=0.9016  test_auc=0.9666


,precision,recall,f1-score,support
0.0,0.883858,0.850379,0.866795,528.000000
1.0,0.911732,0.932571,0.922034,875.000000
accuracy,0.901639,0.901639,0.901639,0.901639
macro avg,0.897795,0.891475,0.894415,1403.000000
weighted avg,0.901242,0.901639,0.901246,1403.000000


Resultados guardados en: /home/albertor/Documents/deep_learning/repos/clasificacion-galaxias/models/experiments/multimodal_token_fusion_attention_m2
